# 09. MDIS 지역별고용조사 B형 원자료 합치기

- 입력: `data/raw/mdis/regional_B_{연도}H{1|2}.csv` (각자 MDIS에서 받은 반기별 파일, 받는 법은 `docs/ai_exposure_design_eda.md` 8-7절)
- 출력: `data/processed/mdis_regional_B.csv` (응답자 1명이 1행)

원자료와 이 출력 파일은 MDIS 준수사항에 따라 저장소에 올리지 않는다(`.gitignore`로 제외됨).
상반기와 하반기의 열 이름이 달라서 상반기 이름으로 맞춘 뒤 이어 붙인다.

In [ ]:
import re
import sys

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
import pandas as pd

from src import config

SRC = config.DATA_RAW / "mdis"
OUT = config.DATA_PROCESSED / "mdis_regional_B.csv"
RENAME = {  # 하반기 이름 → 상반기 이름
    "학력코드": "교육정도_학력구분코드",
    "현직장종사상지위코드": "종사상지위코드",
    "조사대상주간_주된활동상태코드": "활동상태코드",
    "경제활동구분코드": "경제활동인구상태코드",
    "시도가중값": "시도전국가중값",
}
files = sorted(SRC.glob("regional_B_*.csv"))
assert files, f"{SRC}에 regional_B_*.csv 파일이 없음"

## 1. 열 목록 정하기

파일마다 열이 조금 다르다(`직장종사자규모코드`는 2021년부터, `이직횟수코드`는 하반기에만 있음). 한 번이라도 나온 열을 모두 쓰고, 없는 파일에서는 빈칸으로 둔다.

In [ ]:
columns = ["period"]
for f in files:
    for c in pd.read_csv(f, encoding="cp949", dtype=str, nrows=0).rename(columns=RENAME).columns:
        if c not in columns:
            columns.append(c)
print(len(files), "개 파일,", len(columns), "개 열")

## 2. 이어 붙이기

파일을 하나씩 읽어 바로 쓴다(전체를 한꺼번에 메모리에 올리지 않기 위해). `period`는 다른 표와 같은 형식이다(`202501` = 2025 상반기).

In [ ]:
summary = []
for i, f in enumerate(files):
    year, half = re.search(r"_(\d{4})H(\d)", f.name).groups()
    df = pd.read_csv(f, encoding="cp949", dtype=str, low_memory=False).rename(columns=RENAME)
    df.insert(0, "period", f"{year}0{half}")
    df.reindex(columns=columns).to_csv(OUT, mode="w" if i == 0 else "a", header=i == 0, index=False, encoding="utf-8")
    summary.append({"period": f"{year}0{half}", "조사연월": ",".join(df["조사연월"].unique()), "행": len(df)})
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))
print("합계", f"{summary['행'].sum():,}행 →", OUT)

## 3. 검사 — 합친 파일을 다시 읽어 KOSIS 공표값과 비교

가중값을 더한 청년(15~29세) 취업자가 반기마다 KOSIS `DT_1ES3B28J`와 1% 안에서 맞아야 한다.

In [ ]:
use = ["period", "만연령", "경제활동인구상태코드", "시도전국가중값"]
chk = pd.read_csv(OUT, usecols=use, dtype={"period": str, "경제활동인구상태코드": str})
assert len(chk) == summary["행"].sum(), "행 수가 원본 합계와 다름"
assert chk[use].notna().all().all(), "핵심 열에 빈칸이 있음"
youth = chk[(chk["경제활동인구상태코드"] == "1") & chk["만연령"].between(15, 29)]
mdis = youth.groupby("period")["시도전국가중값"].sum().div(1000).rename("원자료(천명)")
panel = pd.read_csv(config.DATA_PROCESSED / "panel_mid.csv", dtype={"period": str})
kosis = panel.groupby("period").y15_29.sum().rename("KOSIS(천명)")
cmp = pd.concat([mdis, kosis], axis=1).dropna()
cmp["차이(%)"] = 100 * (cmp["원자료(천명)"] / cmp["KOSIS(천명)"] - 1)
print(cmp.round(2).to_string())
assert cmp["차이(%)"].abs().max() < 1, "KOSIS 공표값과 1% 넘게 다름"